# AEI Index Audit — Colab bootstrap

Run top to bottom. Cell 1 sets up, cell 3 is the go/no-go data gate.

**Before you start:** create an empty public GitHub repo `aei-index-audit`, and
put a GitHub personal access token (repo scope) in Colab Secrets under the key
`GH_TOKEN`. Never paste a token into a cell.

## 1. Environment

In [ ]:
!pip install -q numpy pandas matplotlib scipy huggingface_hub pyfixest tqdm pytest

import os, sys, subprocess
from google.colab import userdata

GH_USER, REPO = "vkenned2", "aei-index-audit"
try:
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
    print("No GH_TOKEN. You can work, but not push.")

if not os.path.exists(REPO):
    url = (f"https://{token}@github.com/{GH_USER}/{REPO}.git" if token
           else f"https://github.com/{GH_USER}/{REPO}.git")
    subprocess.run(["git","clone",url], check=True)

%cd {REPO}
sys.path.insert(0, ".")
!git config user.email "you@example.com" && git config user.name "Vishal Kennedy"

## 2. Verify the maths before touching real data. If these fail, stop.

In [ ]:
!python -m pytest tests/ -q

## 3. DATA GATE

Everything downstream needs the four geographic releases to share a schema.
If they diverge, stop and cut scope to a single wave. Analyses 1 to 4 all work
on one wave.

In [ ]:
from huggingface_hub import list_repo_files
import pandas as pd

REPO_ID = "Anthropic/EconomicIndex"
files = list_repo_files(REPO_ID, repo_type="dataset")
for r in sorted({f.split("/")[0] for f in files if "/" in f}):
    geo = [f for f in files if f.startswith(r+"/") and f.endswith(".csv")
           and any(k in f.lower() for k in ("geo","state","country","raw_claude_ai"))]
    print(f"\n{r}: {len(geo)} candidate geography files")
    for g in geo[:8]: print("   ", g)

In [ ]:
# Fill in ONE geography csv per release from the listing above.
TARGETS = {}   # e.g. {"w1": "release_2025_09_15/.../file.csv"}
assert TARGETS, "Fill in TARGETS from the cell above first."

from huggingface_hub import hf_hub_download
schemas = {}
for w, path in TARGETS.items():
    local = hf_hub_download(REPO_ID, path, repo_type="dataset", local_dir="data/raw")
    df = pd.read_csv(local, nrows=5000); schemas[w] = set(df.columns)
    print(f"\n=== {w} :: {path}\ncolumns: {list(df.columns)}")
    for c in df.columns:
        u = df[c].dropna().unique()
        print(f"  {c:24s} {df[c].nunique():>6} distinct  e.g. {list(map(str,u[:5]))}")

common = set.intersection(*schemas.values()) if len(schemas)>1 else set()
print("\nSHARED:", sorted(common))
print("GATE:", "PASS" if len(common)>=4 else "FAIL -> single-wave scope")

## 4. Build the state panel

Output must have raw levels, not shares: `state, wave, conversations, denominator`.

In [ ]:
import numpy as np
from aei_audit import indices as ix, aggregation as ag, uncertainty as un

panel = ...   # TODO: real parse once the gate passes

## 5. Analysis 1 — scale. Theil between/within over the hierarchy.

In [ ]:
rows=[]
sub = panel[panel.wave=="w1"]
for label, mp in [("Census division (k=9)", ag.CENSUS_DIVISION),
                  ("BEA region (k=8)", ag.BEA_REGION),
                  ("Census region (k=4)", ag.CENSUS_REGION)]:
    d = ix.theil_decompose((sub.conversations/sub.denominator).to_numpy(),
                           [mp[s] for s in sub.state],
                           weights=sub.denominator.to_numpy())
    rows.append({"grouping":label, **d})
pd.DataFrame(rows)[["grouping","total","between","within","share_between"]]

## 6. Analysis 2 — zoning. The novel one.

Same k, different boundaries. If the real geography sits mid-distribution, the
headline statistic is substantially a property of the map.

In [ ]:
import matplotlib.pyplot as plt
s = panel[panel.wave=="w1"].set_index("state")
sim = ag.zoning_monte_carlo(s.conversations/s.denominator, s.conversations,
                            s.denominator, k=9, stat_fn=ix.gini, n_sims=2000)
div = ag.aggregate_panel(panel[panel.wave=="w1"], ag.CENSUS_DIVISION)
obs = ix.gini((div.conversations/div.denominator).to_numpy(), div.denominator.to_numpy())

fig, ax = plt.subplots(figsize=(7,4))
ax.hist(sim, bins=60, color="#cfcfcf", edgecolor="none")
ax.axvline(obs, color="#b03a2e", lw=2,
           label=f"Census divisions = {obs:.3f} (pctile {100*(sim<obs).mean():.0f})")
ax.set_xlabel("Gini of AI usage rate, k=9 regions"); ax.set_ylabel("random partitions")
ax.legend(frameon=False); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig("figures/zoning_null_k9.png", dpi=160)

## 7. Analysis 4 — how many published rankings are real?

In [ ]:
iv = un.aui_intervals(s.conversations.to_numpy(), s.denominator.to_numpy(),
                      s.index.tolist(), n_boot=4000)
P = un.rank_stability(s.conversations.to_numpy(), s.denominator.to_numpy(),
                      s.index.tolist(), n_boot=4000).loc[iv.unit, iv.unit]
print(un.summarise_rank_stability(P)["interpretation"]); iv.head(15)

## 8. Push back to GitHub

In [ ]:
!git add -A && git commit -q -m "analysis: scale, zoning, uncertainty" && git push -q
print("pushed")